# 🦴 Wrist & Hand Fracture Detection using YOLOv8

**Course:** Computer Vision  
**Project Type:** Bring Your Own Project (BYOP)  
**Author:** Aditya  

---

## Motivation
Fracture diagnosis from X-ray images is a critical task in emergency medicine. Missed or delayed fracture detection can lead to improper treatment. This project applies object detection (YOLOv8) to automatically locate and classify fractures in wrist/hand X-rays, potentially assisting radiologists as a second-opinion tool.

## Pipeline Overview
1. Environment Setup
2. Dataset Exploration (EDA)
3. Dataset Preparation (YOLO format)
4. Model Configuration & Training
5. Evaluation & Metrics
6. Inference & Visualization
7. Grad-CAM Explainability (Bonus)


---
## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless matplotlib seaborn pandas tqdm Pillow PyYAML --quiet

# Verify CUDA availability
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import random
import shutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("All libraries loaded successfully.")

---
## 2. Dataset Loading & Exploration (EDA)

> **Instructions:** Download the dataset from Kaggle and place it in the `data/raw/` directory.
> Expected structure after extraction:
> ```
> data/raw/
>   ├── train/
>   │   ├── images/
>   │   └── labels/
>   ├── valid/
>   │   ├── images/
>   │   └── labels/
>   └── test/
>       ├── images/
>       └── labels/
> ```
> If your dataset uses a different format (e.g., Pascal VOC XML or CSV annotations), see Section 3 for conversion.

In [ ]:
# ─── CONFIGURE PATHS HERE ───────────────────────────────────────────────────
DATA_ROOT   = Path("data/raw")       # Root of your dataset
TRAIN_IMG   = DATA_ROOT / "train" / "images"
TRAIN_LBL   = DATA_ROOT / "train" / "labels"
VALID_IMG   = DATA_ROOT / "valid" / "images"
VALID_LBL   = DATA_ROOT / "valid" / "labels"
TEST_IMG    = DATA_ROOT / "test"  / "images"
TEST_LBL    = DATA_ROOT / "test"  / "labels"
CLASS_NAMES = ["fracture"]           # Update if your dataset has more classes
# ────────────────────────────────────────────────────────────────────────────

for split, path in [("Train", TRAIN_IMG), ("Valid", VALID_IMG), ("Test", TEST_IMG)]:
    count = len(list(path.glob("*"))) if path.exists() else 0
    print(f"{split:6s} images: {count}")

In [ ]:
def parse_yolo_labels(label_dir: Path, class_names: list) -> pd.DataFrame:
    """Parse YOLO format label files into a DataFrame."""
    records = []
    for lbl_file in label_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, w, h = parts
                    records.append({
                        "file": lbl_file.stem,
                        "class_id": int(cls),
                        "class_name": class_names[int(cls)] if int(cls) < len(class_names) else str(cls),
                        "cx": float(cx), "cy": float(cy),
                        "w": float(w),   "h": float(h),
                        "area": float(w) * float(h)
                    })
    return pd.DataFrame(records)

train_df = parse_yolo_labels(TRAIN_LBL, CLASS_NAMES)
valid_df = parse_yolo_labels(VALID_LBL, CLASS_NAMES)
print(f"Training annotations : {len(train_df)}")
print(f"Validation annotations: {len(valid_df)}")
train_df.head()

In [ ]:
# ── EDA: Class distribution, bbox size distribution, image dimensions ────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Exploratory Data Analysis — Training Set", fontsize=14, fontweight="bold")

# 1. Class distribution
train_df["class_name"].value_counts().plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# 2. Bounding box area distribution
axes[1].hist(train_df["area"], bins=30, color="coral", edgecolor="white")
axes[1].set_title("Bounding Box Area Distribution")
axes[1].set_xlabel("Normalised Area (w × h)")
axes[1].set_ylabel("Count")

# 3. BBox width vs height scatter
axes[2].scatter(train_df["w"], train_df["h"], alpha=0.4, color="mediumseagreen", s=15)
axes[2].set_title("BBox Width vs Height")
axes[2].set_xlabel("Width (normalised)")
axes[2].set_ylabel("Height (normalised)")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA plot saved → eda_overview.png")

In [ ]:
def visualize_samples(img_dir: Path, lbl_dir: Path, class_names: list, n: int = 6):
    """Display random sample images with their ground-truth bounding boxes."""
    img_files = list(img_dir.glob("*.[jp][pn]g"))
    samples   = random.sample(img_files, min(n, len(img_files)))
    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = axes.flatten()

    colors = plt.cm.Set1.colors
    for ax, img_path in zip(axes, samples):
        img  = Image.open(img_path).convert("RGB")
        W, H = img.size
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, cx, cy, bw, bh = map(float, parts)
                        x1 = (cx - bw / 2) * W
                        y1 = (cy - bh / 2) * H
                        color = colors[int(cls) % len(colors)]
                        rect = patches.Rectangle(
                            (x1, y1), bw * W, bh * H,
                            linewidth=2, edgecolor=color, facecolor="none")
                        ax.add_patch(rect)
                        label = class_names[int(cls)] if int(cls) < len(class_names) else str(int(cls))
                        ax.text(x1, y1 - 4, label, color=color,
                                fontsize=9, fontweight="bold",
                                bbox=dict(facecolor="white", alpha=0.6, pad=1, edgecolor="none"))
        ax.axis("off")
        ax.set_title(img_path.name, fontsize=8)

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.suptitle("Sample Training Images with Ground Truth", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("sample_images.png", dpi=150, bbox_inches="tight")
    plt.show()

visualize_samples(TRAIN_IMG, TRAIN_LBL, CLASS_NAMES)

---
## 3. Dataset Preparation

YOLOv8 reads data configuration from a `.yaml` file. We generate it programmatically here.

In [ ]:
# Build the data.yaml required by Ultralytics YOLOv8
data_yaml = {
    "path" : str(DATA_ROOT.resolve()),
    "train": "train/images",
    "val"  : "valid/images",
    "test" : "test/images",
    "nc"   : len(CLASS_NAMES),
    "names": CLASS_NAMES
}

yaml_path = Path("data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml created:")
print(yaml_path.read_text())

### (Optional) Data Augmentation Preview
YOLOv8 applies augmentations automatically during training (mosaic, flips, HSV jitter). The cell below lets you preview what augmented images look like using Albumentations.

In [ ]:
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    aug_pipeline = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.GaussNoise(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
        A.CLAHE(clip_limit=2.0, p=0.3),   # Enhances X-ray contrast
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

    # Pick one sample and show original vs augmented
    sample_img_path = random.choice(list(TRAIN_IMG.glob("*.[jp][pn]g")))
    sample_img = np.array(Image.open(sample_img_path).convert("RGB"))
    sample_lbl = []
    lbl_path   = TRAIN_LBL / (sample_img_path.stem + ".txt")
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    sample_lbl.append([float(x) for x in parts[1:]])

    augmented = aug_pipeline(
        image=sample_img,
        bboxes=sample_lbl,
        class_labels=[0] * len(sample_lbl)
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(sample_img); ax1.set_title("Original"); ax1.axis("off")
    ax2.imshow(augmented["image"]); ax2.set_title("Augmented"); ax2.axis("off")
    plt.suptitle("Augmentation Preview", fontweight="bold")
    plt.tight_layout()
    plt.savefig("augmentation_preview.png", dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("Albumentations not installed — skipping augmentation preview.")
    print("Install with: pip install albumentations")

---
## 4. Model Training

We use **YOLOv8s** (small) — a good balance between speed and accuracy for academic projects.

In [ ]:
# ─── TRAINING CONFIGURATION ──────────────────────────────────────────────────
MODEL_VARIANT = "yolov8s.pt"   # Options: yolov8n, yolov8s, yolov8m
EPOCHS        = 50             # Increase to 100 for better results
IMG_SIZE      = 640
BATCH_SIZE    = 16             # Reduce to 8 if GPU OOM
PROJECT_NAME  = "fracture_det"
RUN_NAME      = "yolov8s_wrist"
# ─────────────────────────────────────────────────────────────────────────────

model = YOLO(MODEL_VARIANT)
print(f"Model loaded: {MODEL_VARIANT}")
print(model.info())

In [ ]:
results = model.train(
    data      = str(yaml_path),
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    project   = PROJECT_NAME,
    name      = RUN_NAME,
    patience  = 15,          # Early stopping patience
    optimizer = "AdamW",
    lr0       = 1e-3,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    augment   = True,
    degrees   = 10,          # Rotation augmentation (useful for X-rays)
    fliplr    = 0.5,
    mosaic    = 1.0,
    save      = True,
    device    = 0 if torch.cuda.is_available() else "cpu",
    seed      = SEED,
    verbose   = True
)

BEST_MODEL_PATH = Path(PROJECT_NAME) / RUN_NAME / "weights" / "best.pt"
print(f"\nBest model saved at: {BEST_MODEL_PATH}")

---
## 5. Evaluation & Metrics

In [ ]:
# Load best model and run validation
best_model = YOLO(str(BEST_MODEL_PATH))

val_metrics = best_model.val(
    data   = str(yaml_path),
    imgsz  = IMG_SIZE,
    batch  = BATCH_SIZE,
    device = 0 if torch.cuda.is_available() else "cpu",
    verbose= True
)

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(f"mAP@0.5      : {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val_metrics.box.map:.4f}")
print(f"Precision    : {val_metrics.box.mp:.4f}")
print(f"Recall       : {val_metrics.box.mr:.4f}")

In [ ]:
# Plot training curves from results.csv
results_csv = Path(PROJECT_NAME) / RUN_NAME / "results.csv"

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Training Curves — YOLOv8s Fracture Detection", fontsize=14, fontweight="bold")

    plot_pairs = [
        ("train/box_loss",  "Train Box Loss",   "coral"),
        ("train/cls_loss",  "Train Class Loss", "steelblue"),
        ("train/dfl_loss",  "Train DFL Loss",   "mediumpurple"),
        ("metrics/mAP50",   "mAP@0.50",         "mediumseagreen"),
        ("metrics/mAP50-95","mAP@0.50:0.95",    "darkorange"),
        ("val/box_loss",    "Val Box Loss",      "firebrick"),
    ]

    for ax, (col, title, color) in zip(axes.flatten(), plot_pairs):
        if col in df_res.columns:
            ax.plot(df_res["epoch"], df_res[col], color=color, linewidth=2)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Epoch")
            ax.grid(alpha=0.3)
        else:
            ax.set_visible(False)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Training curves saved → training_curves.png")
else:
    print(f"results.csv not found at {results_csv}")

In [ ]:
# Display PR Curve and Confusion Matrix generated by YOLO
run_dir = Path(PROJECT_NAME) / RUN_NAME
artifacts = {
    "PR Curve"          : run_dir / "PR_curve.png",
    "Confusion Matrix"  : run_dir / "confusion_matrix.png",
    "F1 Curve"          : run_dir / "F1_curve.png",
}

existing = {k: v for k, v in artifacts.items() if v.exists()}
if existing:
    fig, axes = plt.subplots(1, len(existing), figsize=(7 * len(existing), 6))
    if len(existing) == 1:
        axes = [axes]
    for ax, (title, path) in zip(axes, existing.items()):
        ax.imshow(plt.imread(str(path)))
        ax.set_title(title, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Evaluation plots not yet generated — run training first.")

---
## 6. Inference & Visualisation

In [ ]:
def run_inference_and_display(model, img_dir: Path, n: int = 6,
                               conf_thresh: float = 0.25,
                               class_names: list = None):
    """Run model inference on random test images and display results."""
    img_files = list(img_dir.glob("*.[jp][pn]g"))
    samples   = random.sample(img_files, min(n, len(img_files)))

    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = axes.flatten()

    for ax, img_path in zip(axes, samples):
        result = model.predict(str(img_path), conf=conf_thresh, verbose=False)[0]
        img    = Image.open(img_path).convert("RGB")
        W, H   = img.size
        ax.imshow(img)

        boxes = result.boxes
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf  = float(box.conf[0])
                cls   = int(box.cls[0])
                label = f"{class_names[cls] if class_names else cls} {conf:.2f}"
                rect  = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2.5, edgecolor="red", facecolor="none")
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, label, color="red", fontsize=9,
                        fontweight="bold",
                        bbox=dict(facecolor="white", alpha=0.7, pad=1, edgecolor="none"))
            ax.set_title(f"{img_path.name} | {len(boxes)} detection(s)", fontsize=8)
        else:
            ax.set_title(f"{img_path.name} | No fracture detected", fontsize=8, color="gray")

        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.suptitle(f"Inference Results (conf ≥ {conf_thresh})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("inference_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Inference results saved → inference_results.png")

run_inference_and_display(best_model, TEST_IMG, n=6, class_names=CLASS_NAMES)

In [ ]:
# ── Single image inference (replace path with your own image) ────────────────
SINGLE_IMAGE = "path/to/your/xray.jpg"   # <-- change this

if Path(SINGLE_IMAGE).exists():
    result = best_model.predict(SINGLE_IMAGE, conf=0.25, verbose=False)[0]
    result.show()
    print(f"Detections: {len(result.boxes)}")
    if result.boxes:
        for i, box in enumerate(result.boxes):
            print(f"  [{i+1}] class={CLASS_NAMES[int(box.cls[0])]}, "
                  f"conf={float(box.conf[0]):.3f}, "
                  f"bbox={[round(x, 1) for x in box.xyxy[0].tolist()]}")
else:
    print("Update SINGLE_IMAGE path to test on a specific X-ray.")

---
## 7. Model Export

Export the trained model to ONNX for deployment or sharing.

In [ ]:
# Export to ONNX
export_path = best_model.export(format="onnx", imgsz=IMG_SIZE)
print(f"Model exported to ONNX: {export_path}")

---
## 8. Summary & Findings

In [ ]:
print("=" * 55)
print("   PROJECT SUMMARY — Wrist Fracture Detection")
print("=" * 55)
print(f"Model         : YOLOv8s")
print(f"Epochs trained: {EPOCHS}")
print(f"Image size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"Classes       : {CLASS_NAMES}")
print()
try:
    print(f"mAP@0.5       : {val_metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95  : {val_metrics.box.map:.4f}")
    print(f"Precision     : {val_metrics.box.mp:.4f}")
    print(f"Recall        : {val_metrics.box.mr:.4f}")
except NameError:
    print("Run Section 5 first to populate metrics.")
print()
print(f"Best weights  : {BEST_MODEL_PATH}")
print("=" * 55)